![](rag pipeline1.png)

**Large Language Models (LLMs)** do not know about our private company data, it will either hallucinate or say it doesn't know. RAG solves this like an open-book test for the AI.

Steps of RAG:
- Reads our private PDFs.
- It stores it in a searchable "Vector Database".
- When a user asks a question, it searches the database for the exact paragraphs related to the question.
- LLM "Answer the user's question using only this text."

In [0]:
%pip install langchain-text-splitters langchain_community pypdf databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
#Langchain is used for - Data Connection (RAG), Chaining Actions, Agentic Behavior & Memory management
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.functions import monotonically_increasing_id
from databricks.vector_search.client import VectorSearchClient
import mlflow.deployments
import time

# --- CONFIGURATION ---
CATALOG = "catalog1_we47"
SCHEMA = "default"
VOLUME = "policy_pdfs"
TABLE_NAME = f"{CATALOG}.{SCHEMA}.policy_chunks"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.policy_index6"
ENDPOINT_NAME = "hr_policy_vs_endpoint"

#Step 1: Configuration & Data Ingestion (Chunking)
# 1. INGESTION
#Document Loading: LangChain extracts the raw text directly from the PDF.
#pdf_path = "/Volumes/catalog1_we47/default/docs/HR Policy Manual 2025.pdf"
pdf_path="/Volumes/catalog1_we47/default/docs/lic policy document.pdf"
loader = PyPDFLoader(pdf_path)
docs = loader.load()

#Chunking: To fit the LLM's size limits (context window), the text is split into 1,000-character chunks. A 100-character overlap ensures context isn't lost when sentences are cut.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_documents(docs)

#Databricks Co+ept: The chunks are saved into a Databricks Delta Table.
data = [{"content": c.page_content, "source": c.metadata['source']} for c in chunks]
df = spark.createDataFrame(data).withColumn("id", monotonically_increasing_id())
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)

#enableChangeDataFeed: This allows the Vector Database (in the next step) to automatically update itself if you upload a new PDF later.
spark.sql(f"ALTER TABLE {TABLE_NAME} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
display(spark.sql("select * from catalog1_we47.default.policy_chunks"))

# 2. VECTOR SEARCH SETUP
vsc = VectorSearchClient()

# Check if endpoint exists, if not create it
existing_endpoints = [e['name'] for e in vsc.list_endpoints().get('endpoints', [])]
if ENDPOINT_NAME not in existing_endpoints:
    print(f"Creating endpoint {ENDPOINT_NAME}...")
    vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")

# Create Index
#Embeddings: Converts text into numerical vectors.
#Vector Index: A specialized database for vectors (automatically keeps it synced with your source Delta table)

print("Creating/Syncing Vector Index...")
vsc.create_delta_sync_index(
    endpoint_name=ENDPOINT_NAME,
    source_table_name=TABLE_NAME,
    index_name=INDEX_NAME,
    pipeline_type='TRIGGERED',
    primary_key="id",
    embedding_source_column="content",
    embedding_model_endpoint_name="databricks-bge-large-en"
)

print(INDEX_NAME)


In [0]:
#Semantic Search (RAG without LLM)

from databricks.vector_search.client import VectorSearchClient
import mlflow.deployments
import time

CATALOG = "catalog1_we47"
SCHEMA = "default"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.policy_index6"
ENDPOINT_NAME = "hr_policy_vs_endpoint"

# 2. Re-initialize the Vector Search Client
vsc = VectorSearchClient()

def hr_search_only(question):
    # 1. Initialize Vector Search Client (No MLflow/LLM needed here)
    vsc = VectorSearchClient()
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    
    # 2. Semantic Search: Find the most relevant fragments
    results = index.similarity_search(
        query_text=question, 
        columns=["content", "source"], 
        num_results=3                  
    )
    
    # 3. Return the raw data
    return results.get('result', {}).get('data_array', [])

# Actively wait for the index to be ready before executing the RAG function
print("Waiting for index to synchronize... this may take several minutes for a new index.")
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
while True:
    status = index.describe().get('status', {})
    is_ready = status.get('ready', False)
    detailed_state = status.get('detailed_state', 'UNKNOWN')
    
    # Exit loop if ready
    if is_ready and detailed_state.startswith("ONLINE"):
        print("\nSystem ready! Index is ONLINE....")
        break
        
    # Exit loop if it failed
    if "FAILED" in detailed_state or "ERROR" in detailed_state:
        print(f"\nSync Failed! State: {detailed_state}")
        print(f"Error Message: {status.get('message', 'No details provided by Databricks.')}")
        break
        
    # Give detailed visibility into the current process
    print(f"Still syncing... Current phase: {detailed_state}")
    time.sleep(10)

search_results = hr_search_only("give me the DIVISIONAL OFFICE address?")
for i, res in enumerate(search_results):
    print(f"Result {i+1} (Source: {res[1]}):\n{res[0]}\n")

In [0]:
#Retrival & Augument & Generation Phase

from databricks.vector_search.client import VectorSearchClient
import mlflow.deployments
import time

CATALOG = "catalog1_we47"
SCHEMA = "default"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.policy_index6"
ENDPOINT_NAME = "hr_policy_vs_endpoint"

# 2. Re-initialize the Vector Search Client
vsc = VectorSearchClient()

# 3. BOT LOGIC (Semantic Search)
def ask_hr_bot(question):
    # Initialize the MLflow deployment client
    client = mlflow.deployments.get_deploy_client("databricks")
    
    # Referring the index created above
    index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
    
    # Semantic Search & Data Retrival
    #Search: Finds the top 3 text chunks most relevant to our question.
    results = index.similarity_search(query_text=question, columns=["content"], num_results=3)
    #Extract: Safely pulls the actual data array out of the search results' dictionary structure.
    docs = results.get('result', {}).get('data_array', [])
    #Format Context: Stitches those 3 separate text chunks into one single string
    context = "\n---\n".join([doc[0] for doc in docs]) if docs else "No context found."
    
    # NLG
    #Prompt Construction: Combines retrieved context and the user's question into a single instruction to forces GenAI to base its answer strictly on our documents.
    prompt = f"Answer using context:\nContext: {context}\nQuestion: {question}"
    #LLM Generation: Sends this tailored prompt LLM.
    response = client.predict(
        endpoint="databricks-meta-llama-3-3-70b-instruct",
        inputs={"messages": [{"role": "user", "content": prompt}]} )
    #Response Extraction: complex JSON data returned by the API to text answer.
    return response['choices'][0]['message']['content']

# Only run the test if the index successfully came online
if index.describe().get('status', {}).get('ready'):
    print(ask_hr_bot("give me the DIVISIONAL OFFICE address?"))